In [1]:
import pandas as pd

# Load the CSV file
file_path = r"/Users/saurabhlevin/Deployment/IDS-DRR-Uttar-Pradesh-Risk-Score-Model/RiskScoreModel/data/risk_score_final_district.csv"
print("Loading CSV file from:", file_path)
df = pd.read_csv(file_path)
print("CSV file loaded successfully. Shape:", df.shape)

Loading CSV file from: /Users/saurabhlevin/Deployment/IDS-DRR-Uttar-Pradesh-Risk-Score-Model/RiskScoreModel/data/risk_score_final_district.csv
CSV file loaded successfully. Shape: (5460, 64)


In [2]:
df.columns


Index(['sdtname', 'object-id', 'st-area(shape)', 'district', 'timeperiod',
       'total-tender-awarded-value',
       'immediate-measures-tenders-awarded-value',
       'others-tenders-awarded-value',
       'repair-and-restoration-tenders-awarded-value', 'max-rain', 'mean-rain',
       'count', 'sum-rain', 'inundation-pct', 'inundation-intensity-mean',
       'inundation-intensity-mean-nonzero', 'inundation-intensity-sum',
       'mean-sex-ratio', 'sum-aged-population', 'sum-young-population',
       'sum-population', 'schools-count', 'total-rail-length', 'rail-count',
       'total-road-length', 'health-centres-count', 'stname', 'dtname',
       'st-area-sh', 'st-length-', 'remarks', 'dist-lgd', 'state-lgd',
       'subdt-lgd', 'ac-no', 'st-length(shape)', 'elevation-mean',
       'slope-mean', 'stcode11', 'dtcode11', 'sub-district-code', 'urban-hhd',
       'urban-electricity', 'urban-tele', 'urban-hhd-pipe',
       'urban-no-sanitation', 'rural-hhd', 'total-hhd', 'avg-electricity'

In [3]:
# Selecting only numeric columns along with 'object-id', 'timeperiod', and 'financial-year'
numeric_columns = df.select_dtypes(include=["number"]).columns
#list(numeric_columns).remove("shape-area")
#numeric_columns = numeric_columns.drop("shape-area")
df_numeric = df[["object-id", "timeperiod", "financial-year"] + list(numeric_columns)]

# Identifying columns that are not included
excluded_columns = [col for col in df.columns if col not in df_numeric.columns]
print("Excluded columns:", excluded_columns)

df_numeric.head()

Excluded columns: ['sdtname', 'district', 'stname', 'dtname']


,object-id,timeperiod,financial-year,st-area(shape),total-tender-awarded-value,immediate-measures-tenders-awarded-value,others-tenders-awarded-value,repair-and-restoration-tenders-awarded-value,max-rain,mean-rain,...,exposure,flood-hazard,government-response,vulnerability,total-tender-awarded-value-fy-cumsum,repair-and-restoration-tenders-awarded-value-fy-cumsum,immediate-measures-tenders-awarded-value-fy-cumsum,others-tenders-awarded-value-fy-cumsum,topsis-score,risk-score
0,09-154-00808,2025_04,2025-2026,9.695190e+08,0.0,0.0,0.0,0.0,0.11,0.07,...,2,5,5,5,0.0,0.0,0.0,0.0,0.794103,5
1,09-153-00799,2025_04,2025-2026,1.703091e+09,0.0,0.0,0.0,0.0,0.08,0.06,...,2,5,5,5,0.0,0.0,0.0,0.0,0.794103,5
2,09-193-00979,2025_04,2025-2026,6.871614e+08,0.0,0.0,0.0,0.0,0.10,0.07,...,2,5,5,4,0.0,0.0,0.0,0.0,0.779183,5
3,09-154-00807,2025_04,2025-2026,1.026696e+09,0.0,0.0,0.0,0.0,0.11,0.05,...,1,5,5,5,0.0,0.0,0.0,0.0,0.742386,5
4,09-190-00963,2025_04,2025-2026,4.477571e+08,0.0,0.0,0.0,0.0,0.10,0.07,...,1,5,5,5,0.0,0.0,0.0,0.0,0.742386,5


In [4]:

df_melted = df_numeric.melt(id_vars=["object-id", "timeperiod", "financial-year"], var_name="factor", value_name="score")
print("Data melted. Shape:", df_melted.shape)

df_melted.head()

Data melted. Shape: (311220, 5)


,object-id,timeperiod,financial-year,factor,score
0,09-154-00808,2025_04,2025-2026,st-area(shape),9.695190e+08
1,09-153-00799,2025_04,2025-2026,st-area(shape),1.703091e+09
2,09-193-00979,2025_04,2025-2026,st-area(shape),6.871614e+08
3,09-154-00807,2025_04,2025-2026,st-area(shape),1.026696e+09
4,09-190-00963,2025_04,2025-2026,st-area(shape),4.477571e+08


In [5]:

df_transposed = df_melted.pivot(index=["factor", "timeperiod", "financial-year"], columns="object-id", values="score").reset_index()
print("Data pivoted. Shape:", df_transposed.shape)
df_transposed.head()
# Save or display the transformed data
output_file = r"/Users/saurabhlevin/Deployment/IDS-DRR-Uttar-Pradesh-Risk-Score-Model/RiskScoreModel/data/Transformed_UP_Data.csv"
df_transposed.to_csv(output_file, index=False)
print("Transformed data saved to:", output_file)

Data pivoted. Shape: (798, 393)
Transformed data saved to: /Users/saurabhlevin/Deployment/IDS-DRR-Uttar-Pradesh-Risk-Score-Model/RiskScoreModel/data/Transformed_UP_Data.csv


In [7]:
# Verify with the source data. Given factor, district, timeperiod and object-id, the score should match

factor = "sdrf-sanctions-awarded-value"
district = '02-031-00202'
timeperiod = '2024_07'
df_transposed.head()
print("modified")
print(df_transposed[(df_transposed["factor"] == factor) & (df_transposed["timeperiod"] == timeperiod) ][[district, "timeperiod", "financial-year", "factor"]])
print("original")
df[(df["object-id"] == district) & (df["timeperiod"] == timeperiod)][['timeperiod', 'financial-year', factor]]



modified


KeyError: "['02-031-00202'] not in index"

In [8]:
dff = pd.read_csv("Transformed_Assam_Data.csv")
conditions = []
operator_map = {
            '==': lambda col, val: col == val,
            '!=': lambda col, val: col != val,
            '>': lambda col, val: col > val,
            '<': lambda col, val: col < val,
            '>=': lambda col, val: col >= val,
            '<=': lambda col, val: col <= val,
            'in': lambda col, val: col.isin(val),
            'not in': lambda col, val: ~col.isin(val)
        }
conditions.append(operator_map["=="](dff["factor"], "sdrf-sanctions-awarded-value"))

dff = dff[pd.concat(conditions, axis=1).all(axis=1)]
dff

,factor,timeperiod,financial-year,18-300,18-300-00101,18-300-00102,18-300-00103,18-300-00104,18-300-00105,18-300-00106,...,18-799,18-799-00124,18-799-00125,18-799-00265,18-816,18-816-00235,18-816-00236,18-816-00258,18-816-00259,18-816-00261
2835,sdrf-sanctions-awarded-value,2021_04,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2836,sdrf-sanctions-awarded-value,2021_05,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2837,sdrf-sanctions-awarded-value,2021_06,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2838,sdrf-sanctions-awarded-value,2021_07,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2839,sdrf-sanctions-awarded-value,2021_08,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2840,sdrf-sanctions-awarded-value,2021_09,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2841,sdrf-sanctions-awarded-value,2021_10,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2842,sdrf-sanctions-awarded-value,2021_11,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2843,sdrf-sanctions-awarded-value,2021_12,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2844,sdrf-sanctions-awarded-value,2022_01,2021-2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
